# YOLO Dataset Creation & Training

**2-class model** — `person` (0) and `ball` (1). Role assignment (team A/B, goalkeeper, referee) is handled in post-processing.

600 annotated frames across 6 matches — 480 train / 120 val.

In [1]:
import sys
import importlib
import torch
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

import src.dataset
importlib.reload(src.dataset)

from src.config import Config
from src.dataset import create_data_yaml, split_val, visualize_labels

dataset_dir = Config.PROJECT_ROOT / "dataset"
train_imgs = len(list((dataset_dir / "images" / "train").glob("*.jpg")))
val_imgs = len(list((dataset_dir / "images" / "val").glob("*.jpg"))) if (dataset_dir / "images" / "val").exists() else 0
print(f"Dataset: {dataset_dir}")
print(f"  Total: {train_imgs + val_imgs} frames ({train_imgs} train, {val_imgs} val)")
if torch.cuda.is_available():
    cc_major, cc_minor = torch.cuda.get_device_capability(0)
    device_arch = f"sm_{cc_major}{cc_minor}"
    supported_arches = set(torch.cuda.get_arch_list())
    gpu_name = torch.cuda.get_device_name(0)
    print(f"CUDA: enabled ({gpu_name})")
    if device_arch in supported_arches:
        print(f"PyTorch CUDA arch support: OK ({device_arch})")
    else:
        print(f"PyTorch CUDA arch support: MISSING ({device_arch})")
        print("Install a newer PyTorch build (CUDA 12.8/13.0) to use GPU for training.")
else:
    print("CUDA: not available (training will use CPU)")

Dataset: C:\Users\PC\Desktop\GitHub\football-computer-vision\dataset
  Total: 600 frames (480 train, 120 val)
CUDA: enabled (NVIDIA GeForce RTX 5070)
PyTorch CUDA arch support: OK (sm_120)


## Train

In [2]:
import os, torch
from ultralytics import YOLO
from src.config import Config
from src.dataset import split_val, create_data_yaml

dataset_dir = Config.PROJECT_ROOT / "dataset"

create_data_yaml(dataset_dir)
split_val(dataset_dir, val_ratio=0.2)

for cache in (dataset_dir / "labels").rglob("*.cache"):
    cache.unlink()
    print(f"Deleted stale cache: {cache.name}")

# ── Training setup ───────────────────────────────────────────────────────────
torch.backends.cudnn.benchmark = True
if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")

use_cuda = False
if torch.cuda.is_available():
    cc_major, cc_minor = torch.cuda.get_device_capability(0)
    if f"sm_{cc_major}{cc_minor}" in set(torch.cuda.get_arch_list()):
        use_cuda = True
device  = 0 if use_cuda else "cpu"
workers = max(2, min(8, os.cpu_count() or 2))

# batch=8 is well-utilised on 12 GB VRAM with yolov8m + imgsz=1280 + AMP.
# autobatch (-1) underestimates peak VRAM when mosaic=1.0 because mosaic
# tiles 4 images per sample, so actual usage is ~4x the clean-forward estimate.
batch = 8 if use_cuda else 2

print(f"Device: {'CUDA' if use_cuda else 'CPU'}")

# ── Model choice ─────────────────────────────────────────────────────────────
# n=3M params (~10 min)  s=11M (~35 min)  m=25M (~60 min)  l=43M (~2 h)
MODEL = "yolov8m.pt"

model = YOLO(MODEL)
model.train(
    data          = str(dataset_dir / "data.yaml"),
    epochs        = 150,
    imgsz         = 1280,
    batch         = batch,
    device        = device,
    workers       = workers,
    mosaic        = 1.0,
    copy_paste    = 0.3,
    cache         = True,
    amp           = use_cuda,
    deterministic = False,
    name          = f"football_2class_{MODEL.replace('.pt', '')}_v1",
    patience      = 30,
)

Created C:\Users\PC\Desktop\GitHub\football-computer-vision\dataset\data.yaml
Split: 480 train, 120 val (from 600 total, 20% split)
Deleted stale cache: train.cache
Deleted stale cache: val.cache
Device: CUDA
Ultralytics 8.4.30  Python-3.13.12 torch-2.12.0.dev20260327+cu128 CUDA:0 (NVIDIA GeForce RTX 5070, 12227MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\PC\Desktop\GitHub\football-computer-vision\dataset\data.yaml, degrees=0.0, deterministic=False, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000014659A4CE50>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0